# Lisbon hotels: vendor-rate ladder and the free tracked-hotel index

Two runs of
[Google Hotels Prices](https://apify.com/yappman/google-hotels-prices):

1. **Search** Lisbon for a 2-night stay 30 days out, with each hotel's per-vendor
   rates, and compare the ladder: who is cheapest, and how wide the spread is.
2. **Tracked** mode, which is free: the index of hotels and stay dates the Actor
   snapshots every week. It shows which Lisbon hotels have price history.

**Cost at Apify Free-tier prices:** at most **$0.0775** in total.

| Run | Events | Most it can cost |
|---|---|---|
| Search, 15 hotels | 15 x $0.002 hotel + 15 x $0.0025 vendor rates + $0.005 start | $0.0725 |
| Tracked index | $0.005 start only, no Actor events | $0.005 |
| History (optional, off) | up to 5 x $0.01 + $0.005 start | $0.055 |

More data: [data.yappman.com](https://data.yappman.com).

In [ ]:
import os
from datetime import date, timedelta
from decimal import Decimal

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from apify_client import ApifyClient

TOKEN = os.environ.get("APIFY_TOKEN")
if not TOKEN:
    raise RuntimeError("Set the APIFY_TOKEN environment variable first. See the README.")
client = ApifyClient(TOKEN)

OUTPUT = Path("output")
OUTPUT.mkdir(exist_ok=True)  # git-ignored: nothing you collect is committed


def run_actor(actor_id: str, run_input: dict, max_charge_usd: float):
    """Start an Actor, wait for it, and return the finished run.

    Spend is capped twice: maxItems in run_input limits the rows saved, and
    max_total_charge_usd stops the run if its charges would pass the cap.
    """
    run = client.actor(actor_id).call(
        run_input=run_input,
        max_total_charge_usd=Decimal(str(max_charge_usd)),
    )
    if run is None or run.status != "SUCCEEDED":
        raise RuntimeError(f"{actor_id} did not succeed: {None if run is None else run.status}")
    print(f"{actor_id}: run {run.id}")
    print("Charged events:", run.charged_event_counts)
    return run

## 1. Search with vendor rates

In [ ]:
ACTOR = "yappman/google-hotels-prices"
check_in = date.today() + timedelta(days=30)
check_out = check_in + timedelta(days=2)
CURRENCY = "EUR"

# Free tier: 15 x ($0.002 + $0.0025) + $0.005 = $0.0725
search_run = run_actor(
    ACTOR,
    {
        "mode": "search",
        "searches": [{"destination": "Lisbon", "checkIn": check_in.isoformat(),
                      "checkOut": check_out.isoformat(), "adults": 2, "rooms": 1}],
        "includeVendorRates": True,
        "currency": CURRENCY,
        "maxItems": 15,
    },
    max_charge_usd=0.08,
)
hotels = pd.DataFrame(client.dataset(search_run.default_dataset_id).iterate_items())
hotels["leadPerNight"] = hotels["leadPrice"].apply(lambda p: (p or {}).get("perNight"))
hotels["leadVendor"] = hotels["leadPrice"].apply(lambda p: (p or {}).get("vendor"))
hotels[["hotelId", "name", "hotelClass", "rating", "reviewCount", "leadPerNight", "leadVendor"]]

## 2. The vendor-rate ladder

One row per hotel and vendor, cheapest first. Sponsored offers are kept and
flagged. Prices are Google's pre-tax figures in the chosen currency.

In [ ]:
ladder = pd.DataFrame([
    {"hotelId": h.hotelId, "hotel": h.name, **v}
    for h in hotels.itertuples() for v in (h.vendorRates or [])
])
ladder = ladder.dropna(subset=["perNight"]).sort_values(["hotel", "perNight"])
ladder["rank"] = ladder.groupby("hotelId")["perNight"].rank(method="min").astype(int)
ladder.to_csv(OUTPUT / "lisbon_vendor_ladder.csv", index=False)

spread = ladder.groupby(["hotelId", "hotel"]).agg(
    vendors=("vendor", "nunique"),
    cheapest=("perNight", "min"),
    dearest=("perNight", "max"),
    cheapestVendor=("vendor", "first"),
).reset_index()
spread["spreadPct"] = ((spread["dearest"] / spread["cheapest"] - 1) * 100).round(1)
spread = spread.merge(hotels[["hotelId", "leadPerNight", "leadVendor"]], on="hotelId")
spread["leadIsCheapest"] = spread["leadPerNight"] <= spread["cheapest"]
spread.sort_values("spreadPct", ascending=False)

In [ ]:
# How often each vendor is cheapest (ties count for every vendor tied)
ladder[ladder["rank"] == 1]["vendor"].value_counts().rename("timesCheapest").head(10)

In [ ]:
top = spread.sort_values("spreadPct", ascending=False).head(10)
sub = ladder[ladder["hotelId"].isin(top["hotelId"])]
order = list(top["hotel"])[::-1]

fig, ax = plt.subplots(figsize=(9, 0.45 * len(order) + 1.5))
for y, name in enumerate(order):
    rows = sub[sub["hotel"] == name]
    ax.plot([rows["perNight"].min(), rows["perNight"].max()], [y, y], color="0.8", zorder=1)
    ax.scatter(rows["perNight"], [y] * len(rows), s=28, zorder=2)
ax.set_yticks(range(len(order)), [n[:40] for n in order])
ax.set_xlabel(f"Per night ({CURRENCY}), {check_in:%d %b} for 2 nights")
ax.set_title("Lisbon: vendor-rate ladder, widest spreads")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()

## 3. The free tracked-hotel index

`tracked` mode charges no Actor events, only the $0.005 start. It writes the
list to the `TRACKED_HOTELS` record of the run's key-value store, not to the
dataset. Each tracked city has two stays a week, 14 and 42 days out, for 2
adults and 2 nights.

In [ ]:
tracked_run = run_actor(ACTOR, {"mode": "tracked"}, max_charge_usd=0.01)
record = client.key_value_store(tracked_run.default_key_value_store_id).get_record("TRACKED_HOTELS")
index = pd.DataFrame(record["value"]["items"])
print(record["value"]["count"], "tracked hotel-stays, generated", record["value"]["generatedAt"])

lisbon = index[index["destination"].str.contains("Lisbon", case=False, na=False)]
print(lisbon["hotelId"].nunique(), "Lisbon hotels tracked")
hotels["tracked"] = hotels["hotelId"].isin(lisbon["hotelId"])
print(hotels["tracked"].sum(), "of the searched hotels have weekly history")
lisbon.head()

## 4. Optional: read stored history (paid, off by default)

`history` mode returns weekly lead-price snapshots for tracked hotels. It costs
$0.01 per hotel with two or more snapshots, and one-snapshot rows are free. It
ignores `maxItems`, so the cost is capped by sending at most five hotels and by
`max_total_charge_usd`. History is in USD for 2 adults.

In [ ]:
RUN_HISTORY = False

if RUN_HISTORY and not lisbon.empty:
    stay = lisbon.iloc[0]
    ids = lisbon[(lisbon["checkIn"] == stay["checkIn"]) & (lisbon["checkOut"] == stay["checkOut"])]["hotelId"]
    # Free tier: at most 5 x $0.01 + $0.005 = $0.055
    hist_run = run_actor(
        ACTOR,
        {"mode": "history", "hotels": list(ids.head(5)),
         "dates": {"checkIn": stay["checkIn"], "checkOut": stay["checkOut"]}},
        max_charge_usd=0.06,
    )
    hist = pd.DataFrame([
        {"hotel": h["name"], "readAt": s["readAt"],
         "perNight": (s.get("leadPrice") or {}).get("perNight"), "deltaPerNight": s.get("deltaPerNight")}
        for h in client.dataset(hist_run.default_dataset_id).iterate_items() for s in h.get("history") or []
    ])
    display(hist)

## Notes

* Rows belong to the Actor's data terms, not to this repository's MIT licence.
  The CSV is written to `output/`, which git ignores.